In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# .env 파일에서 환경 변수 로드
load_dotenv()

# Pinecone 클라이언트 초기화
pinecone_api_key = os.getenv("PINECONE_API_KEY")
if not pinecone_api_key:
    raise ValueError("PINECONE_API_KEY가 .env 파일에 설정되지 않았습니다.")
    
pc = Pinecone(api_key=pinecone_api_key)

print("✅ 라이브러리 및 Pinecone 클라이언트 초기화 완료")

c:\Users\qwer8\anaconda3\envs\rag_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 라이브러리 및 Pinecone 클라이언트 초기화 완료


In [3]:
# 사용할 임베딩 모델을 로드합니다.
# 처음 실행 시 모델 파일을 다운로드하므로 시간이 걸릴 수 있습니다.
model_name = 'BM-K/KoSimCSE-roberta-multitask'
model = SentenceTransformer(model_name)

# 모델의 벡터 차원(dimension) 확인
embedding_dim = model.get_sentence_embedding_dimension()
print(f"✅ 임베딩 모델 '{model_name}' 로드 완료 (벡터 차원: {embedding_dim})")

No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


✅ 임베딩 모델 'BM-K/KoSimCSE-roberta-multitask' 로드 완료 (벡터 차원: 768)


In [4]:
index_name = "rag-assignment"

# 기존에 같은 이름의 인덱스가 있다면 삭제
if index_name in pc.list_indexes().names():
    print(f"⚠️ 기존 인덱스 '{index_name}'를 삭제합니다.")
    pc.delete_index(index_name)

# 새 인덱스 생성
print(f"⏳ 새 인덱스 '{index_name}'를 생성합니다...")
pc.create_index(
    name=index_name,
    dimension=embedding_dim,  # 임베딩 모델의 벡터 차원
    metric="cosine",          # 유사도 측정 방식 (코사인 유사도)
    spec=ServerlessSpec(
        cloud='aws',
        region='us-east-1'
    )
)

# 인덱스 정보 확인
print(f"✅ 인덱스 생성 완료!")
print(pc.describe_index(index_name))

⚠️ 기존 인덱스 'rag-assignment'를 삭제합니다.
⏳ 새 인덱스 'rag-assignment'를 생성합니다...
✅ 인덱스 생성 완료!
{'deletion_protection': 'disabled',
 'dimension': 768,
 'host': 'rag-assignment-m28wzml.svc.aped-4627-b74a.pinecone.io',
 'metric': 'cosine',
 'name': 'rag-assignment',
 'spec': {'serverless': {'cloud': 'aws', 'region': 'us-east-1'}},
 'status': {'ready': True, 'state': 'Ready'},
 'tags': None,
 'vector_type': 'dense'}


In [5]:
# Pinecone 인덱스 객체 가져오기
index = pc.Index(index_name)

# 1단계에서 생성한 청크 데이터 로드
df = pd.read_parquet('../data/corpus_chunks.parquet')

print(f"📄 총 {len(df)}개의 청크를 임베딩하고 Pinecone에 업로드합니다.")

batch_size = 32 # 한 번에 처리할 데이터 개수

for i in tqdm(range(0, len(df), batch_size)):
    # 데이터 배치 선택
    batch_df = df.iloc[i:i+batch_size]
    
    # 텍스트 청크를 임베딩
    texts = batch_df['text'].tolist()
    embeddings = model.encode(texts, convert_to_tensor=True, show_progress_bar=False).tolist()
    
    # Pinecone에 업로드할 데이터 준비 (id, vector, metadata)
    vectors_to_upsert = []
    for idx, row in batch_df.iterrows():
        # 메타데이터는 검색 결과와 함께 반환될 중요한 정보입니다.
        # Pinecone은 너무 많은 메타데이터 필드를 허용하지 않으므로 필요한 것만 선택합니다.
        metadata = {
            'title': row['title'],
            'source': row['source'],
            'url': row['url']
        }
        vectors_to_upsert.append((row['chunk_id'], embeddings[idx-i], metadata))
        
    # Pinecone에 배치 업로드
    index.upsert(vectors=vectors_to_upsert)

print("\n✅ 모든 데이터의 임베딩 및 Pinecone 업로드 완료!")

# 최종 인덱스 상태 확인
print("\n[최종 인덱스 정보]")
print(index.describe_index_stats())

📄 총 52272개의 청크를 임베딩하고 Pinecone에 업로드합니다.


100%|██████████| 1634/1634 [5:56:43<00:00, 13.10s/it]  



✅ 모든 데이터의 임베딩 및 Pinecone 업로드 완료!

[최종 인덱스 정보]
{'dimension': 768,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 52272}},
 'total_vector_count': 52272,
 'vector_type': 'dense'}
